# 07. Agent 기본 실습

- Agent는 LLM이 스스로 Tool을 선택하고 실행하며 목표를 달성함
- LangGraph의 `create_react_agent`를 사용함
- **ReAct 패턴**: Reasoning(추론) → Acting(행동) → Observation(관찰) 반복

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. Tool 준비

In [2]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다."""
    return a * b

@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다."""
    weather_data = {"서울": "맑음, 22도", "부산": "흐림, 18도", "제주": "비, 16도"}
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

tools = [add, multiply, get_current_time, get_weather]

## 2. Agent 생성
- create_agent() 함수 : Tool 사용이 가능한 Agent 그래프를 빠르게 만들어주는 함수
- create_react_agent() 이후 버전에 삭제 될 예정임

In [10]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 1. LLM 생성
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 2. Agent 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 유용한 AI 어시스턴트입니다. 필요한 경우 Tool을 사용해 정확한 정보를 제공하세요."
)

print("Agent 생성 완료")

Agent 생성 완료


- 시퀀스 다이어그램
```mermaid
sequenceDiagram
    participant User as 사용자
    participant Agent as AI Agent
    participant LLM as LLM
    participant Tool as Tool

    User->>Agent: 질문 입력
    Agent->>LLM: 질문 + Tool 목록 전달

    LLM->>LLM: 질문 분석
    LLM-->>Agent: Tool 호출 요청 반환

    Agent->>Tool: Tool 실행
    Tool-->>Agent: Tool 결과 반환

    Agent->>LLM: Tool 결과 전달
    LLM-->>Agent: 최종 답변 생성

    Agent-->>User: 최종 답변 반환
```

## 3. Agent 실행 - 단순 질문

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "지금 몇시야?"
        }
    ]
})

print(result["messages"][-1].content)

현재 시간은 2026년 5월 31일 19시 45분 32초입니다.


## 4. Agent 실행 - 계산이 필요한 질문

In [12]:
result = agent.invoke({"messages": [("human", "25 곱하기 4는 얼마야?")]})

print(result["messages"][-1].content)

25 곱하기 4는 100입니다.


## 5. Agent 실행 - 복합 질문 (여러 Tool 사용)

In [13]:
result = agent.invoke({
    "messages": [("human", "서울 날씨를 알려주고, 15 더하기 27도 계산해줘")]
})

print(result["messages"][-1].content)

서울의 날씨는 맑고, 현재 기온은 22도입니다. 그리고 15 더하기 27은 42입니다.


## 6. Agent 실행 과정 확인

중간 단계(추론, Tool 호출, 결과)를 모두 출력합니다.

In [14]:
result = agent.invoke({"messages": [("human", "부산 날씨는 어때?")]})

for msg in result["messages"]:
    role = msg.__class__.__name__
    print(f"[{role}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("  Tool 호출:", msg.tool_calls)
    else:
        print(" ", msg.content)
    print()

[HumanMessage]
  부산 날씨는 어때?

[AIMessage]
  Tool 호출: [{'name': 'get_weather', 'args': {'city': '부산'}, 'id': 'call_zrQgC4rIc1J0oidqhMgBIcRw', 'type': 'tool_call'}]

[ToolMessage]
  흐림, 18도

[AIMessage]
  부산의 날씨는 흐림이며, 기온은 18도입니다.



## 7. 스트리밍으로 Agent 실행

In [16]:
for chunk in agent.stream({"messages": [("human", "지금 시간과 서울 날씨 알려줘")]}):
    # print(f"새로운 응답 청크 도착: {chunk}")
    for key, value in chunk.items():
        if "messages" in value:
            last_msg = value["messages"][-1]
            print(f"[{key}] {last_msg.content or last_msg.tool_calls}")

새로운 응답 청크 도착: {'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 145, 'total_tokens': 187, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_10896862fa', 'id': 'chatcmpl-DhY5CK2cfWwvRF3Q0MEYji5wUooR9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e44d8-5723-7b82-9411-87405fb88145-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_gAo45Jay41yAEe3QmBha2LEh', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_OUafD32q7zBZyKJYiKTZo3DS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 42, 'tot